# 10 Explain Final Entity-Resolution Decisions
This notebook creates explainability features for final decisions, decomposes composite scores into weighted parts, estimates SHAP-like contribution values, saves XAI explanations, and visualizes score, gap, and entropy patterns.

In [0]:
# Load final decisions, graph tables, and scoring weights.
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt

final_table = "workspace.entity_resolution_project.company_er_final_decisions"
nodes_table = "workspace.entity_resolution_project.company_er_graph_nodes"
edges_table = "workspace.entity_resolution_project.company_er_graph_edges"
xai_table = "workspace.entity_resolution_project.company_er_xai_explanations"

final_df = spark.table(final_table)
nodes = spark.table(nodes_table)
edges = spark.table(edges_table)

NAME_WEIGHT = 0.60
SEMANTIC_WEIGHT = 0.05
COUNTRY_WEIGHT = 0.20
CITY_WEIGHT = 0.15

In [0]:
# Summarize final decision outcomes and confidence diagnostics.
display(
    final_df
    .groupBy("final_decision", "final_decision_source")
    .count()
    .orderBy(F.desc("count"))
)

display(
    final_df
    .groupBy("final_decision")
    .agg(
        F.count("*").alias("count"),
        F.round(F.avg("final_confidence"), 3).alias("avg_final_confidence"),
        F.round(F.avg("top1_score"), 3).alias("avg_top1_score"),
        F.round(F.avg("score_gap_top1_top2"), 3).alias("avg_gap"),
        F.round(F.avg("entropy_norm"), 3).alias("avg_entropy")
    )
    .orderBy("final_decision")
)

In [0]:
# Decompose composite scores and identify the primary score driver.
xai_df = (
    final_df
    .withColumn("name_score_part", F.coalesce(F.col("name_similarity"), F.lit(0.0)) * F.lit(NAME_WEIGHT))
    .withColumn("semantic_score_part", F.coalesce(F.col("semantic_similarity"), F.lit(0.0)) * F.lit(SEMANTIC_WEIGHT))
    .withColumn("country_score_part", F.coalesce(F.col("country_match"), F.lit(0.0)) * F.lit(COUNTRY_WEIGHT))
    .withColumn("city_score_part", F.coalesce(F.col("city_match"), F.lit(0.0)) * F.lit(CITY_WEIGHT))
    .withColumn(
        "max_score_part",
        F.greatest(
            F.col("name_score_part"),
            F.col("semantic_score_part"),
            F.col("country_score_part"),
            F.col("city_score_part")
        )
    )
    .withColumn(
        "primary_score_driver",
        F.when(F.col("name_score_part") == F.col("max_score_part"), F.lit("name_similarity"))
         .when(F.col("semantic_score_part") == F.col("max_score_part"), F.lit("semantic_similarity"))
         .when(F.col("country_score_part") == F.col("max_score_part"), F.lit("country_match"))
         .when(F.col("city_score_part") == F.col("max_score_part"), F.lit("city_match"))
         .otherwise(F.lit("unknown"))
    )
)

In [0]:
# Calculate SHAP-like feature contributions and save XAI explanations.
feature_means = xai_df.agg(
    F.avg("name_similarity").alias("avg_name_similarity"),
    F.avg("semantic_similarity").alias("avg_semantic_similarity"),
    F.avg("country_match").alias("avg_country_match"),
    F.avg("city_match").alias("avg_city_match")
)

xai_df = (
    xai_df
    .crossJoin(feature_means)
    .withColumn("name_shap_like", (F.col("name_similarity") - F.col("avg_name_similarity")) * F.lit(NAME_WEIGHT))
    .withColumn("semantic_shap_like", (F.col("semantic_similarity") - F.col("avg_semantic_similarity")) * F.lit(SEMANTIC_WEIGHT))
    .withColumn("country_shap_like", (F.col("country_match") - F.col("avg_country_match")) * F.lit(COUNTRY_WEIGHT))
    .withColumn("city_shap_like", (F.col("city_match") - F.col("avg_city_match")) * F.lit(CITY_WEIGHT))
)

(
    xai_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(xai_table)
)

In [0]:
# Aggregate score-part diagnostics by final decision and primary driver.
display(
    xai_df
    .groupBy("final_decision")
    .agg(
        F.count("*").alias("count"),
        F.round(F.avg("name_similarity"), 3).alias("avg_name_similarity"),
        F.round(F.avg("semantic_similarity"), 3).alias("avg_semantic_similarity"),
        F.round(F.avg("country_match"), 3).alias("avg_country_match"),
        F.round(F.avg("city_match"), 3).alias("avg_city_match"),
        F.round(F.avg("name_score_part"), 3).alias("avg_name_score_part"),
        F.round(F.avg("semantic_score_part"), 3).alias("avg_semantic_score_part"),
        F.round(F.avg("country_score_part"), 3).alias("avg_country_score_part"),
        F.round(F.avg("city_score_part"), 3).alias("avg_city_score_part")
    )
    .orderBy("final_decision")
)

display(
    xai_df
    .groupBy("primary_score_driver", "final_decision")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
# Inspect row-level XAI explanations for final decisions.
display(
    xai_df
    .select(
        "left_row_key",
        "left_company_name",
        "right_company_name",
        "final_decision",
        "final_decision_source",
        "final_confidence",
        "primary_score_driver",
        F.round(F.col("name_score_part"), 3).alias("name_score_part"),
        F.round(F.col("semantic_score_part"), 3).alias("semantic_score_part"),
        F.round(F.col("country_score_part"), 3).alias("country_score_part"),
        F.round(F.col("city_score_part"), 3).alias("city_score_part"),
        F.round(F.col("composite_score"), 3).alias("composite_score"),
        F.round(F.col("score_gap_top1_top2"), 3).alias("gap"),
        "llm_decision",
        "llm_confidence",
        "llm_reason",
        "final_reason"
    )
    .orderBy("final_decision", F.desc("final_confidence"))
)

In [0]:
# Review LLM-assisted and manual-review decision explanations.
display(
    xai_df
    .filter(F.col("final_decision_source") == "llm_tiebreaker")
    .groupBy("llm_decision")
    .agg(
        F.count("*").alias("count"),
        F.round(F.avg("llm_confidence"), 3).alias("avg_llm_confidence")
    )
    .orderBy(F.desc("count"))
)

display(
    xai_df
    .filter(F.col("final_decision_source").isin("llm_tiebreaker", "manual_review"))
    .select(
        "left_company_name",
        "right_company_name",
        "final_decision",
        "llm_decision",
        "llm_confidence",
        "llm_reason",
        "final_reason"
    )
    .orderBy("final_decision")
)

In [0]:
# Validate graph coverage while reviewing XAI outputs.
display(nodes.groupBy("node_type").count().orderBy(F.desc("count")))
display(edges.groupBy("edge_type").count().orderBy(F.desc("count")))

edge_node_ids = (
    edges.select(F.col("source_node_id").alias("node_id"))
    .union(edges.select(F.col("target_node_id").alias("node_id")))
    .distinct()
)

missing_edge_nodes = (
    edge_node_ids
    .join(nodes.select("node_id"), on="node_id", how="left_anti")
)

display(missing_edge_nodes)

In [0]:
# Plot final-decision distributions for score, gap, and entropy.
plot_df = (
    xai_df
    .select(
        "final_decision",
        "top1_score",
        "score_gap_top1_top2",
        "entropy_norm",
        "name_similarity",
        "semantic_similarity"
    )
    .toPandas()
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = {
    "MATCH": "#2E8B57",
    "NO_MATCH": "#C76E3A",
    "REVIEW": "#6A5ACD",
}

for status, color in colors.items():
    subset = plot_df[plot_df["final_decision"] == status]
    axes[0].hist(subset["top1_score"].dropna(), bins=25, alpha=0.65, label=status, color=color)
    axes[1].hist(subset["score_gap_top1_top2"].dropna(), bins=25, alpha=0.65, label=status, color=color)
    axes[2].hist(subset["entropy_norm"].dropna(), bins=25, alpha=0.65, label=status, color=color)

axes[0].set_title("Top-1 Score by Final Decision")
axes[1].set_title("Score Gap by Final Decision")
axes[2].set_title("Entropy by Final Decision")

for ax in axes:
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()